# Inteligência Artificial 2025/2026
## Jogo PopOut

**Grupo YZ**
* Sérgio Pinto - up202309160
*
*

### Introdução
Neste notebook, apresentamos o desenvolvimento de uma inteligência artificial capaz de jogar **PopOut**.
Dividindo-se em duas fases:
1. Implementação de uma IA baseada em **Monte Carlo Tree Search (MCTS)**.
2. Geração de datasets e treino de uma **Árvore de Decisão (ID3)**.
Apresentamos as decisões de arquitetura, otimizações de performance e a discussão crítica dos resultados de diferentes experiências.

## 1. O Motor do jogo (PopOut)
Antes de contruirmos os cérebros da IA, desenvolvemos a infraestrutura do jogo. O jogo PopOut requer o manuseamento de duas mecânicas principais: `drop` e `pop`.


In [ ]:
import time
import math
import random
import csv
import os
import matplotlib.pyplot as plt
import ID3 as ID3

class PopOutGame:
    ROWS = 6
    COLS = 7
    EMPTY = '-'

    #Cria o tabuleiro vazio
    def __init__(self):
        self.board = [[self.EMPTY for _ in range(self.COLS)] for _ in range(self.ROWS)]
        self.current_player = 'X'

        self.last_player = None
        self.last_move_type = None

        #Variavel que nos permite cumprir se um estado se repetir 3 vezes, é empate
        self.state_history = {}
        self._record_state() 

    #Função para guardar os estados do tabuleiro
    def _record_state(self):
        state_tuple = tuple(tuple(row) for row in self.board)
        self.state_history[state_tuple] = self.state_history.get(state_tuple, 0) + 1

    #Função que cria uma copia do tabuleiro para se puder usar para os testes da IA
    def clone(self):
        """Cria uma cópia do estado do jogo (útil para algoritmos de pesquisa depois)."""
        new_game = PopOutGame()
        new_game.board = [row[:] for row in self.board]
        new_game.current_player = self.current_player

        #Copiar os novos atributos
        new_game.last_player = self.last_player
        new_game.last_move_type = self.last_move_type
        new_game.state_history = self.state_history.copy()

        return new_game
    
    #Função de troca de jogadores
    def switch_player(self):
        self.current_player = 'O' if self.current_player == 'X' else 'X'
    
    #Verifica se coluna esta cheia
    def is_column_full(self, col):
        return self.board[0][col] != self.EMPTY

    #Verificar se o drop esta dentro dos parametros
    def is_valid_drop(self, col):
        return 0 <= col < self.COLS and not self.is_column_full(col)

    #Verificar se o pop esta dentro dos parametros
    def is_valid_pop(self, col):
        return 0 <= col < self.COLS and self.board[self.ROWS - 1][col] == self.current_player

    
    def drop_piece(self, col):
        """Coloca a peça do jogador atual na posição mais baixa disponível da coluna."""
        if not self.is_valid_drop(col):
            return False

        for row in range(self.ROWS - 1, -1, -1):
            if self.board[row][col] == self.EMPTY:
                self.board[row][col] = self.current_player
                return True
        return False
    
    def pop_piece(self, col):
        """
        Remove a peça de baixo da coluna, se ela pertencer ao jogador atual,
        e empurra as peças acima para baixo.
        """
        if not self.is_valid_pop(col):
            return False

        for row in range(self.ROWS - 1, 0, -1):
            self.board[row][col] = self.board[row - 1][col]
        self.board[0][col] = self.EMPTY
        return True

    def apply_move(self, move_type, col):
        """Aplica uma jogada (drop ou pop) e regista o histórico."""
        success = False
        if move_type == 'drop':
            success = self.drop_piece(col)
        elif move_type == 'pop':
            success = self.pop_piece(col)
        
        if success:
            self.last_player = self.current_player
            self.last_move_type = move_type
            self._record_state()

        return success

    def get_legal_moves(self):
        """
        Retorna lista de jogadas possíveis no formato:
        [('drop', col), ('pop', col), ...]
        """
        moves = []

        for col in range(self.COLS):
            if self.is_valid_drop(col):
                moves.append(('drop', col))
            if self.is_valid_pop(col):
                moves.append(('pop', col))

        return moves

    def board_full(self):
        return all(self.board[0][col] != self.EMPTY for col in range(self.COLS))

    # Função antiga mantida para comparação de tempos de execução
    def check_winner_for(self, player):
        """Verifica se o jogador informado tem 4 em linha."""
        # Horizontal
        for row in range(self.ROWS):
            for col in range(self.COLS - 3):
                if all(self.board[row][col + i] == player for i in range(4)):
                    return True

        # Vertical
        for row in range(self.ROWS - 3):
            for col in range(self.COLS):
                if all(self.board[row + i][col] == player for i in range(4)):
                    return True

        # Diagonal principal (\)
        for row in range(self.ROWS - 3):
            for col in range(self.COLS - 3):
                if all(self.board[row + i][col + i] == player for i in range(4)):
                    return True

        # Diagonal secundária (/)
        for row in range(3, self.ROWS):
            for col in range(self.COLS - 3):
                if all(self.board[row - i][col + i] == player for i in range(4)):
                    return True

        return False
    
    def get_winners(self):
        """
        Verifica vencedores varrendo o tabuleiro uma única vez.
        Retorna um conjunto, por exemplo:
        set()        -> ninguém venceu
        {'X'}        -> X venceu
        {'O'}        -> O venceu
        {'X', 'O'}   -> ambos venceram
        """
        winners = set()
        b = self.board
        empty = self.EMPTY

        # Horizontal
        for row in range(self.ROWS):
            for col in range(self.COLS - 3):
                p = b[row][col]
                if (
                    p != empty
                    and b[row][col + 1] == p
                    and b[row][col + 2] == p
                    and b[row][col + 3] == p
                ):
                    winners.add(p)
                    if len(winners) == 2:
                        return winners

        # Vertical
        for row in range(self.ROWS - 3):
            for col in range(self.COLS):
                p = b[row][col]
                if (
                    p != empty
                    and b[row + 1][col] == p
                    and b[row + 2][col] == p
                    and b[row + 3][col] == p
                ):
                    winners.add(p)
                    if len(winners) == 2:
                        return winners

        # Diagonal principal (\)
        for row in range(self.ROWS - 3):
            for col in range(self.COLS - 3):
                p = b[row][col]
                if (
                    p != empty
                    and b[row + 1][col + 1] == p
                    and b[row + 2][col + 2] == p
                    and b[row + 3][col + 3] == p
                ):
                    winners.add(p)
                    if len(winners) == 2:
                        return winners

        # Diagonal secundária (/)
        for row in range(3, self.ROWS):
            for col in range(self.COLS - 3):
                p = b[row][col]
                if (
                    p != empty
                    and b[row - 1][col + 1] == p
                    and b[row - 2][col + 2] == p
                    and b[row - 3][col + 3] == p
                ):
                    winners.add(p)
                    if len(winners) == 2:
                        return winners

        return winners
    
    def get_game_result(self):
        """Verifica o estado do jogo e aplica as regras do PopOut."""

        winners = self.get_winners()

        # Se um pop cria 4 em linha para ambos, o jogador que fez o pop ganha e o outro é ignorado.
        if 'X' in winners and 'O' in winners:
            if self.last_move_type == 'pop':
                return self.last_player
            else:
                return 'DRAW'

        elif 'X' in winners:
            return 'X'

        elif 'O' in winners:
            return 'O'
        
        # Se o mesmo estado se repete 3 vezes, o jogo é declarado empate.
        current_state = tuple(tuple(row) for row in self.board)
        if self.state_history.get(current_state, 0) >= 3:
            return 'DRAW'

        #Verifica se não há mais movimentos possíveis (empate por bloqueio)
        if self.board_full() and len(self.get_legal_moves()) == 0:
            return 'DRAW'
        

        return None


### 1.1. Otimização de Performance: Verificação de Tabuleiro
Durante o desenvolvimento do jogo, identificámos que a função responsável por verificar as vitórias (`check_winner_for`) era o maior problema de performance, pois varria a totalidade do tabuleiro duas vezes consecutivas (uma para 'X' outra para 'O').

Para resolver isto, otimizámos o processo criando a função `get_winners`, que realiza uma **verificação única**. Ao passar por cada célula, o algoritmo avalia simultaneamente o estado de ambos os jogadores. Como o algoritmo MCTS executa centenas de milhares de verificações de tabuleiro por turno, esta otimização matemática foi o que tornou o projeto viável em tempo útil.

Abaixo apresentamos o benchmark que prova visualmente o impacto desta alteração:

In [ ]:
import time
import matplotlib.pyplot as plt

def benchmark_winner_checking(iterations=50000):
    game = PopOutGame()
    # Enchemos o tabuleiro com jogadas aleatórias para criar cenários variados
    for col in range(7):
        game.apply_move('drop', col)
        game.switch_player()

    # 1. Testar função antiga
    start_time = time.time()
    for _ in range(iterations):
        # Lógica antida do check_winner_for
        x_wins = game.check_winner_for('X') if hasattr(game, 'check_winner_for') else False
        o_wins = game.check_winner_for('O') if hasattr(game, 'check_winner_for') else False
    tempo_antigo = time.time() - start_time

    # 2. Testar a Função Nova
    start_time = time.time()
    for _ in range(iterations):
        vencedores = game.get_winners()
    tempo_novo = time.time() - start_time

    return tempo_antigo, tempo_novo

# Correr o teste
print("A executar benchmark de performance (50.000 verificações de tabuleiro)...")
tempo_antigo, tempo_novo = benchmark_winner_checking()

# Gráfico
fig, ax = plt.subplots(figsize=(8, 5))
barras = ax.bar(['Dupla Verificação\n(Abordagem Inicial)', 'Verificação Única\n(Otimização get_winners)'], 
                [tempo_antigo, tempo_novo], 
                color=['#e74c3c', '#2ecc71'])

# Adicionar os valores exatos no topo das barras
for barra in barras:
    yval = barra.get_height()
    ax.text(barra.get_x() + barra.get_width()/2, yval + 0.01, f'{yval:.2f}s', ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Tempo de Execução (Segundos)')
ax.set_title('Impacto da Otimização do Algoritmo de Verificação de Vitórias')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.show()

print(f"Resultado: A otimização tornou o código {tempo_antigo/tempo_novo:.1f}x mais rápido!")

    

## 2. Monte Carlo Tree Search (MCTS)
A nossa implementação MCTS utiliza a fórmula **Upper Confidence Bound for Trees (UCT)** para equilibrar a exploração.

In [ ]:
#Monte Carlo Tree Search
import math
import random

class Node:
    def __init__(self, state, parent=None, move=None):
        self.state = state      #estado atual do jogo
        self.parent = parent    #nó pai
        self.move = move        #jogada que originou este estado

        self.children = []      #lista de nós filhos já expandidos
        self.wins = 0           #numero de vitorias a partir de este nó
        self.visits = 0         #numero de vezes que este no foi visitado
        self.untried_moves = state.get_legal_moves()

        if parent is not None:
            self.player_who_moved = parent.state.current_player
        else:
            self.player_who_moved = None

        #Guardamos as jogadas possiveis que ainda nao transformamos em filhos
        self.untried_moves = state.get_legal_moves()

    def uct_value(self, c_param=1.41):
        #Calcula o valor UCT do nó para decidir se devemos explorá-lo

        #Se o nó nunca foi visitado, damos prioridade maxima para o explorar
        if self.visits == 0:
            return float('inf')
        
        #Taxa de vitoria deste nó
        tx_v = self.wins / self.visits

        #Incentiva a explorar nós menos visitados face ao pai
        exp = c_param * math.sqrt(math.log(self.parent.visits) / self.visits)

        return tx_v + exp
    
    def add_child(self, child_state, move):
        #Cria um filho e junta-o à arvore
        child_node = Node(state=child_state, parent=self, move=move)
        self.untried_moves.remove(move)
        self.children.append(child_node)
        return child_node
    
    def update(self, result):
        #Atualiza estatísticas após uma simulação
        self.visits +=1

        if result == self.player_who_moved:
            self.wins += 1
        elif result == 'DRAW':
            #Um empate é melhor do que derrota logo adicionamos 0.5
            self.wins += 0.5

class MCTS:
    def __init__(self, ai_player, iterations=3000):
        self.ai_player = ai_player      #Define se a IA está a jogar com 'X' ou 'O'
        self.iterations = iterations    #Quantos "jogos à sorte" vamos simular por jogada
        self.root = None

    def update_root(self, move):
        """Avisa a IA da ultima jogada feita para nao criar sempre uma nova todas as vezes"""
        if self.root is not None:
            #Procura nos filhos da raiz atual a jogada que acabou de acontecer 
            for child in self.root.children:
                if child.move == move:
                    self.root = child
                    self.root.parent = None #Corta a ligação para trás dessa
                    return
        self.root = None

    def search(self, initial_state):
        #Devolve a melhor jogada possível após realizar milhares de simulações

        #Nó raiz estado do tabuleiro exatamente como ele está agora
        #So criamos se não tivermos um
        if (self.root is None
            or self.root.state.board != initial_state.board
            or self.root.state.current_player != initial_state.current_player):
            self.root = Node(state=initial_state.clone())

        #O ciclo de simulações(limitado pelo iteration 1000)
        for _ in range(self.iterations):
            node = self.root
            state = initial_state.clone()   #Usamos um clone novo para podermos usar nas simulações

            #Descemos pela árvore já explorada usando a fórmula UCT
            while len(node.untried_moves) == 0 and len(node.children) > 0:
                node = max(node.children, key=lambda c: c.uct_value())
                state.apply_move(node.move[0], node.move[1])
                state.switch_player()  #Alternamos o jogador para refletir a mudança de estado

            #Se chegarmos a um no com jogadas por testar
            if len(node.untried_moves) > 0 and state.get_game_result() is None:
                #Escolhemos uma jogada à sorte que ainda nao testamos
                move = random.choice(node.untried_moves)
                state.apply_move(move[0], move[1])
                state.switch_player()  #Alternamos o jogador para refletir a mudança de estado
                #Criamos um filho novo na árvore
                node = node.add_child(state.clone(), move)
            
            #SIMULAÇÃO
            #Jogamos sempre à sorte até alguem ganhar ou empatar
            while state.get_game_result() is None:
                possible_moves = state.get_legal_moves()
                if not possible_moves:
                    break
                move = random.choice(possible_moves)
                state.apply_move(move[0], move[1])
                state.switch_player()  #Alternamos o jogador para refletir a mudança de estado
            
            #No fim da simulação vemos que ganhou
            result = state.get_game_result()

            while node is not None:
                node.update(result)
                node = node.parent
        
        #A jogada que escolhemos é do filho que foi visitado mais vezes
        best_child = max(self.root.children, key=lambda c: c.visits)
        return best_child.move


### 2.1. Otimização de Memória (Reutilização da Árvore)
Uma implementação *standard* de MCTS sofre de "amnésia" a cada turno, reconstruindo a árvore do zero. Para otimizar o tempo e aumentar a inteligência da IA, implementámos um sistema de atualização da raiz (`update_root`).
Isto permite que a IA reaproveite os nós já explorados (tanto pelas suas jogadas como pelas do adversário), o que nos permitiu atingir um nível de inteligência elevado usando menos iterações e menos tempo de CPU.

### 2.2. Geração do Dataset
Para treinar a nossa Árvore de Decisão, precisávamos de dados, criamos então a função (`generate_dataset`) que põe duas IA's MCTS a jogar uma contra a outra.

Realizámos duas experiências principais de hiperparâmetros para perceber a relação entre Quantidade e Qualidade:
* **Dataset Rápido (100 iterações):** Mais jogos, mas com decisões de baixa qualidade e erros táticos.
* **Dataset Inteligente (3000 iterações):** Menos jogos, mas com decisões táticas irrepreensíveis graças à otimização da árvore de pesquisa.

In [ ]:
def generate_dataset(self, num_games=50, iterations=3000):

        """Cronómetro temporario para teste de tempos de execução após mudanças"""
        tempo_inicio = time.time()

        """
        Duas IAs a jogar uma contra a outra e guarda as 
        decisões num ficheiro CSV
        """
        print(f"A iniciar a geração de {num_games} jogos (iterações: {iterations})")
        
        filename = f"dataset_popout_{iterations}.csv"

        ficheiro_existe = os.path.isfile(filename)

        # 1. Abrir o ficheiro CSV em modo de escrita ('w')
        with open(filename, mode='a', newline='') as file:
            writer = csv.writer(file)
            
            # Só escrevemos o cabeçalho (a primeira linha do Excel/CSV) se o ficheiro acabou de ser criado
            if not ficheiro_existe:
                writer.writerow(['Board_State', 'Best_Move'])

            # 2. O Ciclo dos Jogos
            for i in range(num_games):
                print(f"A simular jogo {i + 1} de {num_games}...")
                
                # Cria um tabuleiro novo e limpo para esta partida
                sim_game = PopOutGame()
                
                # Cria os dois "cérebros" com as tuas 3000 iterações
                ia_X = MCTS.MCTS(ai_player='X', iterations=iterations)
                ia_O = MCTS.MCTS(ai_player='O', iterations=iterations)

                # 3. O Ciclo de Turnos (jogar até o jogo acabar)
                while sim_game.get_game_result() is None:
                    

                    if(sim_game.current_player == 'X'):
                        best_move = ia_X.search(sim_game) 
                    else:
                        best_move = ia_O.search(sim_game)
                    
                    
                    estado_tabuleiro = str(sim_game.board)
                    jogada = f"{best_move[0]} {best_move[1]}"
                    
                    writer.writerow([estado_tabuleiro, jogada])
                    
                    # Aplicar a jogada no tabuleiro
                    sim_game.apply_move(best_move[0], best_move[1])

                    # Avisar as duas ia's que a jogada aconteceu
                    # Assim avançam a árvore e não começam do zero
                    ia_X.update_root(best_move)
                    ia_O.update_root(best_move)
                    #ia_X.root = None
                    #ia_O.root = None
                    
                    # Trocar de jogador para o próximo turno.
                    sim_game.switch_player()

        tempo_fim = time.time()
        tempo_total = tempo_fim - tempo_inicio

        print(f"\nGeração concluída! Ficheiro '{filename}' criado/atualizado com sucesso.")
        print(f"Tempo Total de execução {tempo_total:.2f}")


## 3. Decision Trees (ID3)
Conforme estipulado, implementamos o algoritmo ID3 do zero, sem recurso ao `scikit-learn` para o treino da árvore.

### 3.1. Dataset Iris
Antes de aplicar o ID3 ao PopOut, validámos a implementação num problema clássico e bem conhecido: o dataset Iris (150 flores, 4 medições, 3 espécies).
O ID3 foi concebido para atributos categóricos, mas o Iris tem valores contínuos por isso o primeiro passo é a **discretização**

#### Carregamento e Discretização
Convertemos os valores contínuos em categorias (`≤t`, `t1-t2`, `>t`) usando pontos de corte calculados por ganho de informação — ou seja, os cortes são feitos nos sítios onde as espécies mudam.

In [ ]:
import csv
import math
from collections import Counter

def load_csv(filepath):
    with open(filepath, newline='') as f:
        reader = csv.DictReader(f)
        data = []
        for row in reader:
            data.append({
                'sepallength': float(row['sepallength']),
                'sepalwidth':  float(row['sepalwidth']),
                'petallength': float(row['petallength']),
                'petalwidth':  float(row['petalwidth']),
                'class':       row['class']
            })
    return data

def find_best_thresholds(data, feature):
    sorted_data = sorted(data, key=lambda x: x[feature])
    candidates = []
    for i in range(len(sorted_data) - 1):
        if sorted_data[i]['class'] != sorted_data[i+1]['class']:
            midpoint = (sorted_data[i][feature] + sorted_data[i+1][feature]) / 2
            candidates.append(round(midpoint, 2))
    candidates = sorted(set(candidates))
    best_thresholds = []
    best_gain = -1
    def entropy_of_split(thresholds):
        buckets = assign_buckets(data, feature, thresholds)
        bucket_classes = {}
        for row, bucket in zip(data, buckets):
            bucket_classes.setdefault(bucket, []).append(row['class'])
        total = len(data)
        return sum((len(c)/total)*entropy(c) for c in bucket_classes.values())
    base_entropy = entropy([r['class'] for r in data])
    for c in candidates:
        gain = base_entropy - entropy_of_split([c])
        if gain > best_gain:
            best_gain = gain
            best_thresholds = [c]
    for i in range(len(candidates)):
        for j in range(i+1, len(candidates)):
            gain = base_entropy - entropy_of_split([candidates[i], candidates[j]])
            if gain > best_gain:
                best_gain = gain
                best_thresholds = [candidates[i], candidates[j]]
    return best_thresholds

def assign_buckets(data, feature, thresholds):
    labels = []
    for row in data:
        val = row[feature]
        if len(thresholds) == 0:
            labels.append('all')
        elif len(thresholds) == 1:
            t = thresholds[0]
            labels.append(f'\u2264{t}' if val <= t else f'>{t}')
        else:
            t1, t2 = thresholds[0], thresholds[1]
            if val <= t1:   labels.append(f'\u2264{t1}')
            elif val <= t2: labels.append(f'{t1}-{t2}')
            else:           labels.append(f'>{t2}')
    return labels

def discretize(data, features):
    thresholds_map = {}
    for feature in features:
        thresholds = find_best_thresholds(data, feature)
        thresholds_map[feature] = thresholds
        print(f'  {feature}: cortes em {thresholds}')
    discretized = []
    for row in data:
        new_row = {'class': row['class']}
        for feature in features:
            new_row[feature] = assign_buckets([row], feature, thresholds_map[feature])[0]
        discretized.append(new_row)
    return discretized, thresholds_map

def discretize_example(example, features, thresholds_map):
    return {f: assign_buckets([{f: example[f]}], f, thresholds_map[f])[0] for f in features}

#### Entropia e Ganho de Informação
A **entropia** mede a desordem num grupo (0 = grupo puro; máximo = classes igualmente distribuídas). O **ganho de informação** mede quanto a entropia diminui ao dividir por um atributo — o ID3 escolhe sempre o atributo com maior ganho.

In [ ]:
def entropy(classes):
    total = len(classes)
    if total == 0:
        return 0
    counts = Counter(classes)
    return -sum((c/total)*math.log2(c/total) for c in counts.values() if c > 0)

def information_gain(data, feature):
    total = len(data)
    parent_entropy = entropy([row['class'] for row in data])
    groups = {}
    for row in data:
        groups.setdefault(row[feature], []).append(row['class'])
    weighted = sum((len(c)/total)*entropy(c) for c in groups.values())
    return parent_entropy - weighted

#### Algoritmo ID3
Constrói a árvore recursivamente: em cada nó escolhe o atributo com maior ganho, divide os dados pelos seus valores, e repete para cada subconjunto até todos os exemplos pertencerem à mesma classe ou não existirem mais atributos.

In [ ]:
def build_tree(data, features, depth=0):
    classes = [row['class'] for row in data]
    if len(set(classes)) == 1:
        return {'leaf': classes[0]}
    if not features or not data:
        return {'leaf': Counter(classes).most_common(1)[0][0]}
    gains = {f: information_gain(data, f) for f in features}
    best = max(gains, key=gains.get)
    node = {'feature': best, 'branches': {}}
    remaining = [f for f in features if f != best]
    for value in sorted(set(row[best] for row in data)):
        subset = [row for row in data if row[best] == value]
        node['branches'][value] = build_tree(subset, remaining, depth+1)
    return node

def get_all_leaves(tree):
    if 'leaf' in tree:
        return [tree['leaf']]
    return [l for child in tree['branches'].values() for l in get_all_leaves(child)]

def classify(tree, example):
    if 'leaf' in tree:
        return tree['leaf']
    value = example.get(tree['feature'])
    if value not in tree['branches']:
        return Counter(get_all_leaves(tree)).most_common(1)[0][0]
    return classify(tree['branches'][value], example)

def print_tree(node, prefix='', is_last=True, branch_label=''):
    connector = '\u2514\u2500\u2500 ' if is_last else '\u251c\u2500\u2500 '
    extension = '    ' if is_last else '\u2502   '
    if 'leaf' in node:
        print(f'{prefix}{connector}{branch_label} \u2192 {node["leaf"]}')
    else:
        items = list(node['branches'].items())
        if branch_label:
            print(f'{prefix}{connector}{branch_label}')
            prefix = prefix + extension
        print(f'{prefix}[{node["feature"]}]')
        for i, (val, child) in enumerate(items):
            print_tree(child, prefix, i == len(items)-1, val)

#### Treino, Avaliação e Classificação
Dividimos os dados em 80% treino / 20% teste (estratificado por classe), treinamos a árvore e avaliamos a precisão nos exemplos nunca vistos. Por fim, classificamos 3 flores novas para demonstrar a utilização.

In [ ]:
def train_test_split(data, test_ratio=0.2, seed=42):
    classes = {}
    for i, row in enumerate(data):
        classes.setdefault(row['class'], []).append((i, row))
    train, test = [], []
    for class_data in classes.values():
        n = len(class_data)
        indices = [(i*37+seed)%n for i in range(n)]
        seen, shuffled = set(), []
        for idx in indices:
            if idx not in seen:
                seen.add(idx)
                shuffled.append(class_data[idx])
        for item in class_data:
            if item not in shuffled:
                shuffled.append(item)
        split = int(len(shuffled)*(1-test_ratio))
        train.extend([r for _,r in shuffled[:split]])
        test.extend([r for _,r in shuffled[split:]])
    return train, test

# --- Executar ---
features = ['sepallength', 'sepalwidth', 'petallength', 'petalwidth']
data = load_csv('iris.csv')
train_raw, test_raw = train_test_split(data, test_ratio=0.2)
train_disc, thresholds_map = discretize(train_raw, features)

test_disc = []
for row in test_raw:
    new_row = {'class': row['class']}
    for f in features:
        new_row[f] = assign_buckets([row], f, thresholds_map[f])[0]
    test_disc.append(new_row)

tree = build_tree(train_disc, features)

print('\n=== Árvore de Decisão ID3 ===')
print_tree(tree)

correct = sum(1 for raw, disc in zip(test_raw, test_disc)
              if classify(tree, disc) == disc['class'])
print(f'\nPrecisão: {correct}/{len(test_disc)} ({correct/len(test_disc)*100:.1f}%)')

print('\n=== Classificação de Novos Exemplos ===')
novos = [
    {'sepallength':5.1,'sepalwidth':3.5,'petallength':1.4,'petalwidth':0.2},
    {'sepallength':6.0,'sepalwidth':2.9,'petallength':4.5,'petalwidth':1.5},
    {'sepallength':6.8,'sepalwidth':3.0,'petallength':5.5,'petalwidth':2.1},
]
for ex in novos:
    disc_ex = discretize_example(ex, features, thresholds_map)
    print(f'  PL={ex["petallength"]} PW={ex["petalwidth"]} → {classify(tree, disc_ex)}')

### 3.2. ID3 aplicado ao PopOut
Após validar o algoritmo com o Iris, treinámos a árvore final usando o nosso objetivo principal: o jogo PopOut.

Ao contrário do dataset Iris, que exigiu discretização de valores contínuos, **o estado do tabuleiro do PopOut é categórico**. Cada uma das 42 casas do tabuleiro só pode assumir três valores: `'X'`, `'O'` ou vazia `'-'`.

Isto permitiu-nos utilizar a versão otimizada do algoritmo ID3. Em vez de calcularmos limiares do corte, o algoritmo converte a matriz do tabuleiro (6x7) numa lista unidimensional de 42 atributos e usa o Ganho de Informação de forma direta para ramificar as decisões.

#### 3.2.1. Preparação dos Dados e Entropia
Abaixo apresentamos as funções responsáveis por achatar o estado do jogo e calcular a impureza das decisões no dataset gerado pelo nosso MCTS (`dataset_popout_3000.csv`).

In [ ]:
import csv
import ast
import math
from collections import Counter


def carregar_dados(filename):
    # Lê o CSV e converte a string do tabuleiro numa lista de 42 atributos.

    X = [] # guarda os tabuleiros (cada um com 42 posições)
    y = [] # guarda as melhores jogadas
    
    print(f"A ler o ficheiro {filename}...")
    
    with open(filename, mode='r') as file:
        reader = csv.reader(file)
        next(reader) # Saltar o cabeçalho
        
        for linha in reader:
            if not linha: continue
            
            
            board_matriz = ast.literal_eval(linha[0])
            best_move = linha[1]
            
            # "Achatar" a matriz 6x7 numa única lista de 42 elementos
            board_achatado = []
            for row in board_matriz:
                board_achatado.extend(row)
                
            X.append(board_achatado)
            y.append(best_move)
            
    print(f"Total de exemplos: {len(X)}")
    return X, y

def calcular_entropia(labels):
    #Calcula a impureza/entropia de um conjunto de decisões.

    total_exemplos = len(labels)
    if total_exemplos == 0:
        return 0.0
    
    # Conta quantas vezes cada jogada aparece na lista
    contagem = Counter(labels)
    entropia = 0.0
    
    for jogada, quantidade in contagem.items():
        probabilidade = quantidade / total_exemplos
        # Fórmula da Entropia
        entropia -= probabilidade * math.log2(probabilidade)
        
    return entropia

#### 3.2.2. Ganho de Informação e Construção Recursiva da Árvore
O algoritmo vai testar todas as casas disponíveis do tabuleiro para descobrir qual delas reduz mais o "caos" nas decisões futuras, construindo a árvore recursivamente até obter uma folha pura ou esgotar os atributos.

In [ ]:
def ganho_informacao(x, y, indice):
    entropia_base = calcular_entropia(y)

    subconjuntos = {}
    for i in range(len(x)):
        valor_da_casa = x[i][indice]
        if valor_da_casa not in subconjuntos:
            subconjuntos[valor_da_casa] = []
        subconjuntos[valor_da_casa].append(y[i])
    
    entropia_nova = 0.0
    total_exemplos = len(y)

    for valor_da_casa, jogadas in subconjuntos.items():
        peso = len(jogadas) / total_exemplos
        entropia_nova += peso * calcular_entropia(jogadas)

    return entropia_base - entropia_nova

def construir_arvore(x, y, atributos):
    #Contrutor da arvore passo a passo

    # todas as jogadas neste ramo forem iguais, devolvemos essa jogada (Folha Pura)
    if len(set(y)) == 1:
        return y[0]
    
    # já não houver mais casas para testar, devolvemos a jogada mais comum
    if len(atributos) == 0:
        return Counter(y).most_common(1)[0][0]
    
    # escolher a melhor pergunta
    melhor_ganho = -1
    melhor_atributo = None

    for atributo in atributos:
        ganho = ganho_informacao(x, y, atributo)
        if ganho > melhor_ganho:
            melhor_ganho = ganho
            melhor_atributo = atributo

    # se nenhuma pergunta melhorar a informação, devolvemos a jogada mais comum
    if melhor_ganho == 0:
        return Counter(y).most_common(1)[0][0]
    
    # criar o no da arvore
    arvore = {melhor_atributo: {}}

    # remover o atributo escolhido para não voltarmos a perguntar sobre a mesma casa
    novos_atributos = atributos.copy()
    novos_atributos.remove(melhor_atributo)

    # descobrir que valores ('X', 'O', '-') existem nesta casa no nosso dataset
    valores_nesta_casa = set([x[i][melhor_atributo] for i in range(len(x))])

    for valor in valores_nesta_casa:
        x_sub = []
        y_sub = []

        for i in range(len(x)):
            if x[i][melhor_atributo] == valor:
                x_sub.append(x[i])
                y_sub.append(y[i])
            
        arvore[melhor_atributo][valor] = construir_arvore(x_sub, y_sub, novos_atributos)
    
    return arvore

#### 3.2.3. Jogador e Navegação na Árvore
Para integrar o ID3 no motor do PopOut, criámos a classe `ID3Jogador`. Ela achata o tabuleiro e cada turno e navega nos ramos do dicionário até encontrar uma jogada recomendada. Incluímos também um sistema de tolerância a falhas para cenários nunca antes vistos no dataset.

In [ ]:
def prever_jogada(arvore, estado_tabuleiro):
    if not isinstance(arvore ,dict):
        return arvore
    
    pergunta_casa = list(arvore.keys())[0]

    valor_no_tabuleiro = estado_tabuleiro[pergunta_casa]

    if valor_no_tabuleiro in arvore[pergunta_casa]:
        proximo_no = arvore[pergunta_casa][valor_no_tabuleiro]
        return prever_jogada(proximo_no, estado_tabuleiro)
    else:
        #Cair num cenário que a árvore nunca viu na vida
        ramo_seg = list(arvore[pergunta_casa].keys())[0]
        proximo_no = arvore[pergunta_casa][ramo_seg]
        return prever_jogada(proximo_no, estado_tabuleiro)
    

class ID3Jogador:
    def __init__(self, arvore_treinada):
        self.arvore = arvore_treinada

    def search(self, game_state):
        board_achatado = []
        for row in game_state.board:
            board_achatado.extend(row)

        jogada_str = prever_jogada(self.arvore, board_achatado)

        partes = jogada_str.split()
        move_type = partes[0]
        coluna = int(partes[1])

        return (move_type, coluna)

#### 3.2.4. Resultados do Treino ID3
Abaixo testamos a eficiência do nosso código.

In [ ]:
import time

X_dados, y_jogadas = carregar_dados("dataset_popout_3000.csv")
    
# O nosso tabuleiro tem 42 posições (0 a 41)
atributos = list(range(42))

tempo_inicio = time.time()

# 2. Treinar a Árvore Mágica
minha_arvore_id3 = construir_arvore(X_dados, y_jogadas, atributos)

tempo_fim = time.time()

print(f"Tempo: {tempo_fim - tempo_inicio:.2f} segundos!")
print(f"Raiz da árvore (A primeira pergunta que a IA faz): A casa {list(minha_arvore_id3.keys())[0]}")

## 4. Experiências Finais e Resultados
Para avaliar verdadeiramente o sucesso da nossa arquitetura, colocámos os vários modelos a competir entre si.

1. **MCTS (100 iterações) vs MCTS (1000 iterações):** O impacto do tempo de pensamento.
2. **ID3 (Treinado a 3000 iterações) vs MCTS (Tempo Real):** A derradeira prova entre a árvore estática e o pensador em tempo real.

In [ ]:
import time
import matplotlib.pyplot as plt

def mcts_vs(num_games=5, iter_x=100, iter_o=1000):
    print(f"Jogo entre MCTS Rápido ({iter_x} iter) VS MCTS Forte ({iter_o} iter)")

    vitorias_x = 0
    vitorias_o = 0
    empates = 0

    tempo_inicio = time.time()

    for i in range(num_games):
        game = PopOutGame()
        ia_X = MCTS(ai_player='X', iterations=iter_x)
        ia_O = MCTS(ai_player='O', iterations=iter_o)

        while game.get_game_result() is None:
            if game.current_player == 'X':
                move = ia_X.search(game)
            else:
                move = ia_O.search(game)
            
            game.apply_move(move[0], move[1])

            #Atualizar as memórias
            ia_X.update_root(move)
            ia_O.update_root(move)

            game.switch_player()

        resultado = game.get_game_result()
        if resultado == 'X':
            vitorias_x += 1
            print(f"Jogo {i+1}: Vitória do X (Rápido)")
        elif resultado == 'O':
            vitorias_o += 1
            print(f"Jogo {i+1}: Vitória do O (Forte)")
        else:
            empates += 1
            print(f"Jogo {i+1}: Empate")
    
    tempo_total = time.time() - tempo_inicio
    print(f"\nConcluído em {tempo_total:.1f} segundos.")
    return vitorias_x, vitorias_o, empates

vitorias_x, vitorias_o, empates = mcts_vs(num_games=5, iter_x=100, iter_o=1000)

labels = ['MCTS Rápido (X) - 100 Iterações', 'MCTS Forte (O) - 1000 Iterações', 'Empates']
sizes = [vitorias_x, vitorias_o, empates]
colors = ['#e74c3c', '#2ecc71', '#95a5a6']
explode = (0, 0.1, 0)  # Destaca a "fatia" do vencedor esperado

# 1. Função mágica para esconder o texto se a percentagem for zero
def mostrar_percentagem(pct):
    return f'{pct:.1f}%' if pct > 0 else ''

# Aumentamos um pouco o tamanho da figura para a legenda caber bem
fig1, ax1 = plt.subplots(figsize=(8, 6))

# 2. Retirámos o 'labels=labels' daqui de dentro para limpar o gráfico
# e passámos a nossa nova função ao 'autopct'
fatias, textos, textos_perc = ax1.pie(sizes, explode=explode, colors=colors, autopct=mostrar_percentagem, shadow=True, startangle=90)

# Garantir que o gráfico é um círculo perfeito
ax1.axis('equal')  

# 3. Adicionar a Legenda organizada na parte de baixo
plt.legend(fatias, labels, loc="upper center", bbox_to_anchor=(0.5, -0.05), ncol=1)

plt.title('Taxa de Vitória: Quantidade vs Qualidade de Simulações', pad=20)
plt.tight_layout() # Ajusta as margens para nada ficar cortado
plt.show()


### 4.2. ID3 vs MCTS
Nesta experiência, colocamos os nossos dois algoritmos a competir entre si.

* **Jogador X (ID3):** Conhecimento Estático. Joga praticamente instantâneo, baseando-se nas regras que extraiu do dataset de 3000 iterações.
* **Jogador O (MCTS):** Capacidade de Adaptação. Pensa em tempo real executando 1000 simulações por turno para encontrar a melhor jogada.

In [ ]:
import time
import matplotlib.pyplot as plt


def idvsmc(num_games=5):
     
    X, y = ID3.carregar_dados("dataset_popout_3000.csv")
    arvore = ID3.construir_arvore(X, y, list(range(42)))
    print("Concluído\n")
    
    print(f"ID3 (X) vs MCTS 1000 iter (O)")
    vitorias_id3 = 0
    vitorias_mcts = 0
    empates = 0

    tempo_inicio = time.time()

    for i in range(num_games):
        game = PopOutGame()
        
        ia_X = ID3.ID3Jogador(arvore_treinada=arvore)
        
        ia_O = MCTS(ai_player='O', iterations=1000)

        while game.get_game_result() is None:
            if game.current_player == 'X':
                move = ia_X.search(game)
            else:
                move = ia_O.search(game)

            success = game.apply_move(move[0], move[1])
            
            if not success:
                print(f"⚠️ A IA {game.current_player} tentou uma jogada inválida: {move}")
                break

            ia_O.update_root(move)

            game.switch_player()

        resultado = game.get_game_result()
        if resultado == 'X':
            vitorias_id3 += 1
            print(f"Jogo {i+1}: Vitória do ID3 (Árvore)")
        elif resultado == 'O':
            vitorias_mcts += 1
            print(f"Jogo {i+1}: Vitória do MCTS")
        else:
            empates += 1
            print(f"Jogo {i+1}: Empate")

    tempo_total = time.time() - tempo_inicio
    
    print(f"\nRESULTADOS({num_games} Jogos | Tempo: {tempo_total:.1f}s) ---")
    print(f"ID3 (X): {vitorias_id3} vitórias")
    print(f"MCTS (O): {vitorias_mcts} vitórias")
    print(f"Empates: {empates}")
    
    return vitorias_id3, vitorias_mcts, empates


vitorias_id3, vitorias_mcts, empates = idvsmc(num_games=10)

labels = ['ID3 (Cérebro Estático)', 'MCTS (Pensador Dinâmico)', 'Empates']
sizes = [vitorias_id3, vitorias_mcts, empates]
colors = ['#f39c12', '#2980b9', '#95a5a6']
explode = (0.05, 0.05, 0) # Faz as fatias separarem-se um bocadinho

def mostrar_percentagem(pct):
    return f'{pct:.1f}%' if pct > 0 else ''

fig, ax = plt.subplots(figsize=(8, 5))
fatias, textos, textos_perc = ax.pie(sizes, explode=explode, colors=colors, autopct=mostrar_percentagem, shadow=True, startangle=140)
ax.axis('equal')

plt.legend(fatias, labels, loc="upper center", bbox_to_anchor=(0.5, -0.05), ncol=1)
plt.title('Duelo Final: ID3 vs MCTS', pad=20, fontweight='bold')
plt.tight_layout()
plt.show()

### 5. Conclusão

Este projeto permitiu-nos explorar dois lados muito diferentes da Inteligência Artificial: um algoritmo que tenta adivinhar o futuro simulando milhares de jogos na hora (MCTS), e outro que aprende a jogar estudando o passado (Árvore de Decisão/ID3).

Da análise dos nossos resultados, tiramos  três conclusões:

1. **A Qualidade dos Dados é superior a algoritmos complexos:** No jogo final, o ID3 treinado com dados de um MCTS de 3000 iterações, conseguiu ganhar a um MCTS de 1000 iterações. Conseguindo provar que a nossa Árvore de Decisão conseguiu adquirir inteligência de jogo e ganhar a um algoritmo que pensa na hora.

2. **Rapidez:** O MCTS é superior no aspecto de que consegue-se adaptar a qualquer surpresa no tabuleiro, porém demora mais tempo a pensar em cada jogada. Já o ID3 a escolher a jogada é basicamente instantâneo.

3. **Otimizações de Código:** Otimizamos a função de verificar quem ganhou (`get_winners`), tornando o jogo cerca de 11 vezes mais rápido como demonstrado na secção 1.1.

Provámos que é possivel passar o conhecimento tático de um algoritmo de pesquisa dispendioso (MCTS) para dentro de uma estrutura estática e leve (ID3), mantendo uma competitividade alta entre ambas.